In [1]:
# pip install keras-tuner

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import L1, L2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD, Adam

from kerastuner.tuners import RandomSearch

C:\Users\FGstore\AppData\Local\Temp\ipykernel_16984\2547709895.py:11: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch


In [3]:
df = pd.read_csv(r"C:\Users\FGstore\Downloads\train.csv")

In [4]:
df.dropna(inplace=True)

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
X = df.drop(['Survived', 'Name', 'Ticket', 'Cabin'], axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8,  random_state=42)

In [7]:
cat_cols = list(X_train.select_dtypes(include=['object']).columns)
encoder = OrdinalEncoder()
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols] )
X_test[cat_cols] = encoder.transform(X_test[cat_cols] )

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
X_train.shape[1]

8

In [10]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(64, activation='relu', kernel_regularizer=L1(0.01)),
    Dropout(0.7),
    Dense(32, activation='relu', kernel_regularizer=L2(0.001)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.summary()

C:\Users\FGstore\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,681 (61.25 KB)

 Trainable params: 15,681 (61.25 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile( optimizer=SGD(momentum=0.9),
    # optimizer=Adam(learning_rate=0.0001),
    loss = 'binary_crossentropy', metrics = ['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=5)

model.fit(X_train, y_train, epochs=20,
          validation_split=0.1, batch_size=32, 
          callbacks=[early_stopping])

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.6565 - loss: 5.1227 - val_accuracy: 0.6667 - val_loss: 5.1090
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5649 - loss: 5.1148 - val_accuracy: 0.8000 - val_loss: 4.9563
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6031 - loss: 4.9336 - val_accuracy: 0.8000 - val_loss: 4.7803
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6565 - loss: 4.7247 - val_accuracy: 0.8000 - val_loss: 4.5894
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6565 - loss: 4.5785 - val_accuracy: 0.8000 - val_loss: 4.4016
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6641 - loss: 4.4244 - val_accuracy: 0.8000 - val_loss: 4.2273
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6718 - loss: 4.1933 - val_accuracy: 0.8000 - val_loss: 4.0404
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6718 - loss: 4.0308 - val_accuracy: 0.8000 - val_loss: 3.8576


In [12]:
model.evaluate(X_test, y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6216 - loss: 2.1039


[2.1038520336151123, 0.6216216087341309]

## hyperparams tune

In [13]:
# funcation to find best params
def build_model(hp):
    model = Sequential([
        # hp.Int("units", min_value=32, max_value=256, step=32)
        # units the key
        # min_value = the min value to start with
        # max_value = the max value to end with
        # step is the gap between any choice
            Dense(units=hp.Int("units", min_value=32, max_value=256, step=32), 
                  activation='relu', input_shape=(X_train.shape[1],)),
            
            # hp.Choice('activation', ['relu', 'sigmoid'])
            # choice either relu or sigmoid
            Dense(10, activation=hp.Choice('activation', ['relu', 'sigmoid'])),

            Dense(1, activation="sigmoid")
    ])

    model.compile(
        # learning_rate=hp.Choice('learning_rate', [0.01, 0.001, 0.0001]
        # find any learning rate in list: [0.01, 0.001, 0.0001]
        optimizer=Adam(learning_rate=hp.Choice('learning_rate', [0.01, 0.001, 0.0001])),
        loss = 'binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# find random param
tuner = RandomSearch(
    build_model,  # model arch
    objective='val_loss',  # moniter to check the best model (val_accuracy)
    max_trials=3,  # number of make experiments
    directory = 'lab4',  # dir to save each experiment
    project_name = 'session 6' # folder for this project
)

early_stopping = EarlyStopping(patience=5, 
                               restore_best_weights=True)

checkpoints = ModelCheckpoint("best_model.keras", 
                              monitor="val_loss", 
                              save_best_only=True)

tuner.search(X_train, y_train, 
             epochs=5, 
             validation_split=0.1,
             callbacks=[early_stopping, checkpoints])

Reloading Tuner from lab4\session 6\tuner0.json


In [14]:
# get best model with best params
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

C:\Users\FGstore\AppData\Roaming\Python\Python313\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,237 (4.83 KB)

 Trainable params: 1,237 (4.83 KB)

 Non-trainable params: 0 (0.00 B)